In [ ]:
"""
Obtiene elevación, pendiente (slope) y orientación (aspect) promedio
para un área alrededor de un punto, usando SRTM o Copernicus DEM vía
Google Earth Engine.

CÓMO LEER EL RESULTADO
-----------------------
elevation_m       Elevación promedio del área, en metros sobre el nivel del mar.
elevation_std_m   Cuánto varía la elevación dentro del área. Un valor alto
                   indica terreno con desniveles marcados (no todo a la misma altura).

slope_deg         Pendiente promedio, en grados (0° = plano, 90° = vertical).
                   Como referencia general (no una regla estricta, revisa
                   siempre criterios agronómicos específicos para tu cultivo):
                     0-8°   : plano a suave, fácil manejo
                     8-15°  : moderado, puede requerir prácticas de conservación
                     15-25° : pronunciado, suele necesitar curvas de nivel/terrazas
                     >25°   : muy pronunciado, manejo más difícil/costoso

slope_std_deg     Cuánto varía la pendiente dentro del área. Alto = terreno
                   irregular (mezcla de zonas planas y empinadas); bajo =
                   pendiente consistente en toda el área (el promedio la
                   representa bien).

aspect_deg        Orientación hacia la que "mira" la pendiente, en grados
                   de brújula (0°=Norte, 90°=Este, 180°=Sur, 270°=Oeste).
                   Relevante para exposición solar: en el hemisferio norte,
                   laderas orientadas al sur/suroeste reciben más sol directo.
"""
import math
from datetime import datetime
from pathlib import Path
import pandas as pd
import ee
ee.Initialize()


DEM_SOURCES = {
    "copernicus": "COPERNICUS/DEM/GLO30_2024_1",  # versión 2024, recomendado
    "srtm": "USGS/SRTMGL1_003",                    # clásico, puede tener huecos en zonas escarpadas
}


def get_terrain_profile_area(lat, lon, area_meters=56, dem_source="copernicus", verbose=False):
    """
    Calcula elevación, pendiente y orientación promedio sobre un área
    alrededor del punto. area_meters es el radio del buffer; el área
    real evaluada es un cuadrado de lado 2*area_meters (ver nota en
    save_terrain_profile / la conversación de diseño del pipeline).

    dem_source: "copernicus" (default, recomendado) o "srtm"
    verbose: si True, imprime las estadísticas crudas devueltas por GEE (debug)
    """
    if dem_source not in DEM_SOURCES:
        raise ValueError(f"dem_source debe ser uno de {list(DEM_SOURCES.keys())}")

    center = ee.Geometry.Point([lon, lat])
    region = center.buffer(area_meters).bounds()

    if dem_source == "copernicus":
        # COPERNICUS/DEM/GLO30 es una ImageCollection de tiles -> hay que mosaiquear
        dem = ee.ImageCollection(DEM_SOURCES[dem_source]).select('DEM').mosaic()
    else:
        dem = ee.Image(DEM_SOURCES[dem_source]).select('elevation')

    elevation = dem.rename('elevation')

    # IMPORTANTE: .mosaic() no conserva una proyección/escala de pixel bien
    # definida -> ee.Terrain.slope()/aspect() necesitan una grilla explícita
    # para calcular el gradiente correctamente. Sin esto, terminan usando
    # una escala por defecto mucho más gruesa que 30m.
    elevation_for_terrain = elevation.reproject(crs='EPSG:4326', scale=30)
    slope = ee.Terrain.slope(elevation_for_terrain).rename('slope')
    aspect_deg = ee.Terrain.aspect(elevation_for_terrain)

    # Aspect es un dato circular (0-360°) -> promediamos via seno/coseno,
    # no con una media aritmética directa (eso daría resultados incorrectos
    # cuando el área cruza el norte, ej. valores cerca de 0° y 360°).
    aspect_rad = aspect_deg.multiply(math.pi / 180)
    aspect_sin = aspect_rad.sin().rename('aspect_sin')
    aspect_cos = aspect_rad.cos().rename('aspect_cos')

    combined = elevation.addBands(slope).addBands(aspect_sin).addBands(aspect_cos)

    combined_reducer = (
        ee.Reducer.mean()
        .combine(reducer2=ee.Reducer.stdDev(), sharedInputs=True)
        .combine(reducer2=ee.Reducer.count(), sharedInputs=True)
    )

    stats = combined.reduceRegion(
        reducer=combined_reducer,
        geometry=region,
        scale=30,
        bestEffort=True
    ).getInfo()

    if verbose:
        print(f"[DEBUG] stats crudos: {stats}")

    mean_aspect_rad = math.atan2(stats['aspect_sin_mean'], stats['aspect_cos_mean'])
    mean_aspect_deg = math.degrees(mean_aspect_rad)
    if mean_aspect_deg < 0:
        mean_aspect_deg += 360

    results = {
        'lat': lat,
        'lon': lon,
        'dem_source': dem_source,
        'elevation_m': stats['elevation_mean'],
        'elevation_std_m': stats['elevation_stdDev'],
        'slope_deg': stats['slope_mean'],
        'slope_std_deg': stats['slope_stdDev'],
        'aspect_deg': mean_aspect_deg,
        # Nota: no calculamos un "std" de aspect por ser un dato circular
        # (requeriría varianza circular en vez de stdDev normal).
    }
    return results


def save_terrain_profile(terrain_data, out_prefix="terrain_profile_data", output_dir="../databases"):
    """
    Guarda el perfil de terreno con timestamp en el nombre:
    {out_prefix}-vYYMMDDHHMMSS.csv
    """
    df = pd.DataFrame([terrain_data])

    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.csv"

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    df.to_csv(out_path, index=False)
    print(f"CSV guardado en {out_path} ({df.shape[0]}x{df.shape[1]})")
    return out_path, df


if __name__ == "__main__":
# El Playon         --||     7.4584221918243045,    -73.222052853104
# Finca Matanza     --||     7.300921,              -73.009794
# Sugarcane_COL     --||     3.580109040361371,     -76.31299479308868
# Sugarcane_QLD     --||     -19.689669877950884,   147.22717515914223
    ha = 2
    area_meters = math.sqrt(ha * 10000) / 2
    Latitude=7.4584221918243045
    Longitude=-73.222052853104
    terrain_data = get_terrain_profile_area(Latitude, Longitude, area_meters, dem_source="copernicus")
    out_path, df = save_terrain_profile(terrain_data)
    print(df)

CSV guardado en ../databases/terrain_profile_data-v260805181540.csv (1x8)
        lat        lon  dem_source  elevation_m  elevation_std_m  slope_deg  \
0  7.458422 -73.222053  copernicus   995.242932        16.300007  25.622195   

   slope_std_deg  aspect_deg  
0       4.616097  161.759455  
